# 00 pip

In [7]:
!{sys.executable} -m pip install geopandas
!{sys.executable} -m pip install rasterio
!{sys.executable} -m pip install matplotlib
!{sys.executable} -m pip install statsmodels
!{sys.executable} -m pip install matplotlib_scalebar

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


# 01 landsat data process and HR

In [10]:
import os
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from glob import glob

def clip_rasters_with_shapefile(raster_folder, shapefile_path, output_folder):
    # Ensure the output directory exists
    os.makedirs(output_folder, exist_ok=True)

    # Load the shapefile as a GeoDataFrame
    nyc_shape = gpd.read_file(shapefile_path)

    # Loop through each raster in the raster folder
    for raster_path in glob(os.path.join(raster_folder, '*.tif')):
        with rasterio.open(raster_path) as src:
            raster_crs = src.crs

            # Ensure the shapefile is in the same CRS as the raster
            if nyc_shape.crs != raster_crs:
                nyc_shape = nyc_shape.to_crs(raster_crs)

            # Clip the raster using the shapefile
            out_image, out_transform = mask(src, nyc_shape.geometry, crop=True)

            # Update metadata for the output raster
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs  # Use the raster's CRS for the output
            })

            # Save the clipped raster to the output folder
            output_raster_path = os.path.join(output_folder, f"clipped_{os.path.basename(raster_path)}")
            if os.path.isdir(output_raster_path):
                continue
            with rasterio.open(output_raster_path, "w", **out_meta) as dest:
                dest.write(out_image)

            print(f"Clipped raster saved to: {output_raster_path}")

## 1 裁剪出来需要的文件

In [13]:
# Paths to the folder and shapefile
raster_folder = r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge'
shp_seoul = seoul_boundary_file
output_folder = r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped'
os.makedirs(output_folder, exist_ok=True)

clip_rasters_with_shapefile(raster_folder, shp_seoul, output_folder)

Clipped raster saved to: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_QA_PIXEL.TIF
Clipped raster saved to: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_ST_B10.TIF
Clipped raster saved to: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_QA_PIXEL.TIF
Clipped raster saved to: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_ST_B10.TIF


## 2 计算百分比

In [14]:
import rasterio
import numpy as np
import os

def get_sorted_dates(file_path, suffix="QA_PIXEL.TIF"):
    # Retrieve all file names in the folder
    file_names = os.listdir(file_path)

    # Extract and collect date parts from files matching the specified suffix
    date_parts = []
    for file_name in file_names:
        if file_name.endswith(suffix):
            # Extract the date part from the file name
            date_part = file_name.split('_')[1:-2]
            date_parts.append(date_part)

    # Sort the date parts and return as a list
    sorted_dates = sorted(date_parts)

    # Print the sorted dates
    #print(sorted_dates)
    for date in sorted_dates:
        print(date)

    return sorted_dates

file_path = r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped'
sorted_dates = get_sorted_dates(file_path)

def calculate_pixel_proportion(sorted_dates, base_folder, pixel_values=[21824, 21952]):
    for sorted_data in sorted_dates:
        filename = "_".join(sorted_data)
        data_path = rf'{base_folder}\clipped_{filename}_QA_PIXEL.tif'

        if not os.path.exists(data_path):
            print(f"⚠️ 文件不存在: {data_path}")
            continue

        with rasterio.open(data_path) as src:
            qa_pixel_data = src.read(1)

        non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
        total_non_zero_pixels = non_zero_pixels.size

        unique, counts = np.unique(non_zero_pixels, return_counts=True)
        pixel_counts = dict(zip(unique, counts))

        pixel_proportions = {value: count / total_non_zero_pixels for value, count in pixel_counts.items()}

        print(f"📅 {filename}:")
        for val in pixel_values:
            prop = pixel_proportions.get(val, 0)
            print(f"  - 像素值 {val} 的占比: {prop:.2%}")
        print()

base_folder = r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped'
calculate_pixel_proportion(sorted_dates, base_folder)

['LC08', 'L2SP', '116034', '20200530', '20200820', '02', 'T1']
['LC08', 'L2SP', '116034', '20200919', '20201006', '02', 'T1']
📅 LC08_L2SP_116034_20200530_20200820_02_T1:
  - 像素值 21824 的占比: 93.91%
  - 像素值 21952 的占比: 4.22%

📅 LC08_L2SP_116034_20200919_20201006_02_T1:
  - 像素值 21824 的占比: 94.64%
  - 像素值 21952 的占比: 4.85%



In [15]:
import rasterio
import numpy as np

def find_high_proportion_dates(sorted_dates, input_folder, threshold=0.80, target_value=[21824,21952] ):
    high_proportion_dates = []

    for date in sorted_dates:
        # Path to the QA_PIXEL TIFF file for the current date
        filename = "_".join(date)
        data_path = rf'{input_folder}\clipped_{filename}_QA_PIXEL.tif'

        try:
            with rasterio.open(data_path) as src:
                qa_pixel_data = src.read(1)  # Read the first band data

            # Filter for non-zero pixels
            non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
            total_non_zero_pixels = non_zero_pixels.size

            # Count unique pixel values and calculate proportions
            unique, counts = np.unique(non_zero_pixels, return_counts=True)
            pixel_counts = dict(zip(unique, counts))
            pixel_proportions = {value: count / total_non_zero_pixels for value, count in pixel_counts.items()}

            # Check the proportion of the target values
            proportion_target_value = sum(pixel_proportions.get(val, 0) for val in target_value)
            if proportion_target_value > threshold:
                high_proportion_dates.append((filename, proportion_target_value))
                print(f"{filename} pixel values {target_value} combined proportion: {proportion_target_value:.2%}")


        except FileNotFoundError:
            print(f"File not found: {data_path}")

    return high_proportion_dates

input_folder=r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped'
high_proportion_dates = find_high_proportion_dates(sorted_dates, input_folder)

# Print results
print("\nDates with high target value proportion:")
for date, proportion in high_proportion_dates:
    print(f"{date}: {proportion:.2%}")


LC08_L2SP_116034_20200530_20200820_02_T1 pixel values [21824, 21952] combined proportion: 98.13%
LC08_L2SP_116034_20200919_20201006_02_T1 pixel values [21824, 21952] combined proportion: 99.49%

Dates with high target value proportion:
LC08_L2SP_116034_20200530_20200820_02_T1: 98.13%
LC08_L2SP_116034_20200919_20201006_02_T1: 99.49%


# 用 QA数据清理B10 -> clean data

In [17]:
# import rasterio
# import numpy as np
# import os
# for date, proportion in high_proportion_dates:
#     data_workspace = r"D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped"
#     b_10_data = os.path.join(data_workspace, rf'clipped_{date}_ST_B10.tif')
#     qa_data = os.path.join(data_workspace, rf'clipped_{date}_QA_PIXEL.tif')
#     output_folder = os.path.join(data_workspace,r'clean')
#     os.makedirs(output_folder,exist_ok=True)
#
#     output_data = os.path.join(output_folder, rf'clipped_{date}_B10_cleaned.tif')
#     print("B10 Data Path:", b_10_data)
#     print("QA Data Path:", qa_data)
#     print("Output Data Path:", output_data)
#
#     # Read Band 10 and QA data
#     with rasterio.open(b_10_data) as src:
#         band10 = src.read(1).astype(float)  # Convert data to float type to support NaN
#         profile = src.profile
#
#     with rasterio.open(qa_data) as qa_src:
#         qa = qa_src.read(1)  # Read QA pixel data
#
#     # Define labels for different cloud and shadow conditions
#     label = [1, 21824, 21826, 21888, 21890, 22080, 22144, 22280, 23888, 23952, 24088, 24216, 24344, 24472, 30048, 54596, 54852, 55052]
#     print("Dilated cloud over land QA label:", label[2])
#
#     # Create a mask for clouds and shadows, filtering out cloud and shadow areas
#     valid_labels = [21824, 21952]
#     cloud_mask = ~np.isin(qa, valid_labels)  # mask 出非有效值（云、阴影、水等）
#
#
#     # Set cloud and shadow areas to 0
#     band10[cloud_mask] = 0
#
#     # Save the processed result
#     with rasterio.open(output_data, 'w', **profile) as dst:
#         dst.write(band10, 1)


B10 Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_ST_B10.tif
QA Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_QA_PIXEL.tif
Output Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned.tif
Dilated cloud over land QA label: 21826
B10 Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_ST_B10.tif
QA Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_QA_PIXEL.tif
Output Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned.tif
Dilated cloud over land QA label: 21826


# 换成shapefile

In [18]:
# import os
# import rasterio
# import geopandas as gpd
# from rasterio.features import shapes
# from shapely.geometry import shape
#
# def raster_to_shapefile(raster_files, output_folder):
#     # 确保输出目录存在
#     os.makedirs(output_folder, exist_ok=True)
#
#     for raster_file in raster_files:
#         with rasterio.open(raster_file) as src:
#             # 获取栅格的nodata值
#             nodata_value = src.nodata
#
#             # 提取栅格的形状
#             mask = src.read(1)  # 读取第一个波段的数据
#             results = shapes(mask, mask=mask != nodata_value, transform=src.transform)
#
#             # 将结果转换为 GeoDataFrame
#             shapes_list = []
#             for geom, value in results:
#                 geom = shape(geom)
#                 if geom.is_valid:
#                     shapes_list.append({"geometry": geom, "value": value})
#
#             # 创建 GeoDataFrame
#             gdf = gpd.GeoDataFrame(shapes_list, crs=src.crs)
#
#             # 输出 Shapefile 文件路径
#             output_shapefile = os.path.join(output_folder, os.path.basename(raster_file).replace('.tif', '.shp'))
#
#             # 保存为 Shapefile
#             gdf.to_file(output_shapefile)
#
#             print(f"栅格已转换为 Shapefile 并保存到: {output_shapefile}")
#
# # 输入的栅格文件路径
# normal_files = [
#     r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned.tif'
# ]
# extreme_files = [
#
#     r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned.tif'
# ]
#
# # 合并所有栅格文件
# all_raster_files = normal_files + extreme_files
#
# # 输出文件夹路径
# output_shapefile_folder = r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles'
# os.makedirs(output_shapefile_folder, exist_ok=True)
#
# # 转换栅格文件为 Shapefile
# raster_to_shapefile(all_raster_files, output_shapefile_folder)

KeyboardInterrupt: 

# 得到fid的shape文件

In [2]:
# import geopandas as gpd
# import pandas as pd
# import os
# import glob
#
# def fast_clip_and_assign_fid(input_shp, clip_shp, output_shp):
#     print(f"📂 加载输入数据：{input_shp}")
#     input_gdf = gpd.read_file(input_shp)
#     clip_gdf = gpd.read_file(clip_shp)
#
#     # 确保 CRS 一致
#     if input_gdf.crs != clip_gdf.crs:
#         input_gdf = input_gdf.to_crs(clip_gdf.crs)
#
#     # 在裁剪前先加上 city_id 字段
#     clip_gdf = clip_gdf.reset_index().rename(columns={'index': 'city_id'})
#
#     print("✂️ 正在执行 overlay 精准裁剪...")
#     try:
#         clipped = gpd.overlay(input_gdf, clip_gdf[['city_id', 'geometry']], how='intersection', keep_geom_type=False)
#     except Exception as e:
#         print(f"❌ Overlay 错误: {e}")
#         return
#
#     # 保存结果
#     os.makedirs(os.path.dirname(output_shp), exist_ok=True)
#     clipped.to_file(output_shp, driver='ESRI Shapefile')
#     print(f"✅ 已保存: {output_shp}")
#
# # 输入的栅格转 Shapefile 路径（假设已经将其转化为 Shapefile）
# shapefile_files = [
#     r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned.shp',
#     r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned.shp'
# ]
#
# grid_folder = r'D:\seoul\grids'
# # 输出文件夹路径
# output_folder = r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles_clipped_with_fid'
# # 遍历所有 grid shapefile
# for grid_file in os.listdir(grid_folder):
#     if grid_file.endswith('.shp'):
#         grid_path = os.path.join(grid_folder, grid_file)
#
#         for input_shp in shapefile_files:
#             input_name = os.path.splitext(os.path.basename(input_shp))[0]
#             grid_name = os.path.splitext(os.path.basename(grid_file))[0]
#             output_name = f'{input_name}_clipped_by_{grid_name}_with_fid.shp'
#             output_shp = os.path.join(output_folder, output_name)
#             if os.path.exists(output_shp):
#                 print(f"⏩ 已存在，跳过: {output_shp}")
#                 continue
#             fast_clip_and_assign_fid(input_shp, grid_path, output_shp)

⏩ 已存在，跳过: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles_clipped_with_fid\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned_clipped_by_grid_1080m_with_fid.shp
⏩ 已存在，跳过: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles_clipped_with_fid\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned_clipped_by_grid_1080m_with_fid.shp
⏩ 已存在，跳过: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles_clipped_with_fid\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned_clipped_by_grid_120m_with_fid.shp
⏩ 已存在，跳过: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles_clipped_with_fid\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned_clipped_by_grid_120m_with_fid.shp
⏩ 已存在，跳过: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles_clipped_with_fid\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned_clipped_by_grid_240m_with_fid.sh

# 先将数据洗成摄氏度 bt -> lst

In [6]:
# import rasterio
# import numpy as np
# import os
#
# def calculate_lst(input_files, output_dir, multiplier=0.00341802, add_constant=149.0, nodata_value=-9999):
#     """
#     Calculate Land Surface Temperature (LST) from a list of raster files and save the output.
#
#     Args:
#         input_files (list): List of paths to input raster files.
#         output_dir (str): Directory to save the output raster files.
#         multiplier (float): Multiplication factor for converting to temperature.
#         add_constant (float): Additive constant for converting to temperature.
#         nodata_value (float): Value to use for no-data in the output files.
#     """
#     # Ensure the output directory exists
#     os.makedirs(output_dir, exist_ok=True)
#
#     for input_file in input_files:
#         with rasterio.open(input_file) as src:
#             # Read the first band and convert to float
#             band_data = src.read(1).astype(float)
#             profile = src.profile
#
#         # Replace values <= 0 with NaN
#         band_data[band_data <= 0] = np.nan
#
#         # Calculate brightness temperature (BT)
#         lst_kelvin = (band_data * multiplier) + add_constant
#
#         # Convert to Celsius
#         lst_celsius = lst_kelvin - 273.15
#
#         # Replace NaN values with the nodata value
#         lst_celsius[np.isnan(lst_celsius)] = nodata_value
#
#         # Update profile for the output file
#         profile.update({
#             'dtype': 'float32',
#             'count': 1,
#             'nodata': nodata_value
#         })
#
#         # Generate a short output file name
#         base_name = os.path.basename(input_file).split('.')[0]
#         output_file = os.path.join(output_dir, f"{base_name}_LST.tif")
#
#         # Save the LST raster
#         with rasterio.open(output_file, 'w', **profile) as dst:
#             dst.write(lst_celsius, 1)
#
#         print(f"LST saved to: {output_file}")
#
#
# # 输入的栅格文件路径
# normal_files = [
#     r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned.tif'
# ]
# extreme_files = [
#     r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned.tif'
# ]
#
# # 输出目录
# output_dir_normal = r'D:\seoul\Final_data\landsat_final\data\normal_heat_day_lst'
# output_dir_extreme = r'D:\seoul\Final_data\landsat_final\data\extreme_heat_day_lst'
#
# # 计算并保存 LST
# calculate_lst(normal_files, output_dir_normal)
# calculate_lst(extreme_files, output_dir_extreme)

LST saved to: D:\seoul\Final_data\landsat_final\data\normal_heat_day_lst\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned_LST.tif
LST saved to: D:\seoul\Final_data\landsat_final\data\extreme_heat_day_lst\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned_LST.tif


# 相减，然后得到HR map

In [7]:
import rasterio
import numpy as np
import os

def calculate_heat_resilience(normal_files, extreme_files, output_dir, nodata_value=-9999):
    os.makedirs(output_dir, exist_ok=True)

    if len(normal_files) != len(extreme_files):
        raise ValueError("Number of normal and extreme files must be the same.")

    for normal_file, extreme_file in zip(normal_files, extreme_files):
        with rasterio.open(normal_file) as normal_src, rasterio.open(extreme_file) as extreme_src:
            normal_data = normal_src.read(1).astype(float)
            extreme_data = extreme_src.read(1).astype(float)

            if normal_data.shape != extreme_data.shape:
                raise ValueError(f"Dimension mismatch between {normal_file} and {extreme_file}.")

            # 强制处理 nodata
            if normal_src.nodata is not None:
                normal_data[normal_data == normal_src.nodata] = np.nan
            else:
                normal_data[normal_data == nodata_value] = np.nan

            if extreme_src.nodata is not None:
                extreme_data[extreme_data == extreme_src.nodata] = np.nan
            else:
                extreme_data[extreme_data == nodata_value] = np.nan

            # 差值
            heat_resilience = normal_data - extreme_data

            # 打印调试
            print(f"\n📁 Processing: {os.path.basename(normal_file)} vs {os.path.basename(extreme_file)}")
            print("🔹 First 10 values from normal_data (non-NaN):", normal_data[~np.isnan(normal_data)].flatten()[:10])
            print("🔹 First 10 values from extreme_data (non-NaN):", extreme_data[~np.isnan(extreme_data)].flatten()[:10])
            print("🔸 First 10 values from heat_resilience (non-NaN):", heat_resilience[~np.isnan(heat_resilience)].flatten()[:10])

            # 替换 NaN 为 nodata
            heat_resilience[np.isnan(heat_resilience)] = nodata_value

            profile = normal_src.profile
            profile.update({
                'dtype': 'float32',
                'nodata': nodata_value
            })

            year = os.path.basename(normal_file).split('_')[4][:4]
            output_file = os.path.join(output_dir, f"{year}_heat_resilience.tif")

            with rasterio.open(output_file, 'w', **profile) as dst:
                dst.write(heat_resilience.astype('float32'), 1)

            print(f"✅ Heat resilience raster saved to: {output_file}")




# 输入的栅格文件路径
normal_files = [
    r'D:\seoul\Final_data\landsat_final\data\normal_heat_day_lst\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned_LST.tif'
]
extreme_files = [
    r'D:\seoul\Final_data\landsat_final\data\extreme_heat_day_lst\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned_LST.tif'
]

# 输出目录
output_dir = r'D:\seoul\Final_data\landsat_final\data\heat_resilience'

# 计算并保存 heat resilience
calculate_heat_resilience(normal_files, extreme_files, output_dir, nodata_value=-9999)


📁 Processing: clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned_LST.tif vs clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned_LST.tif
🔹 First 10 values from normal_data (non-NaN): [22.07631302 22.79067993 22.37709999 22.72915459 22.42153358 23.30680084
 23.11539078 23.31705475 23.60074997 23.68278313]
🔹 First 10 values from extreme_data (non-NaN): [26.02754402 26.52999306 25.89424133 26.16084671 26.15401077 26.85128784
 26.57101059 26.64962387 26.75558281 26.70431328]
🔸 First 10 values from heat_resilience (non-NaN): [-3.951231   -3.73931313 -3.51714134 -3.43169212 -3.73247719 -3.544487
 -3.45561981 -3.33256912 -3.15483284 -3.02153015]
✅ Heat resilience raster saved to: D:\seoul\Final_data\landsat_final\data\heat_resilience\2020_heat_resilience.tif


# 计算面积和占比

In [3]:
# import geopandas as gpd
# import os
# import pandas as pd
# from concurrent.futures import ThreadPoolExecutor
# def add_lst_ratio_within_only(clipped_gdf, city_path, output_path):
#     # ✅ STEP 1: 读取 grid 文件（即 city_gdf），用其 CRS 作为标准
#     city_gdf = gpd.read_file(city_path)
#     target_crs = city_gdf.crs  # ⬅️ 使用它为主
#     clipped = clipped_gdf.to_crs(target_crs)
#     city_gdf = city_gdf.to_crs(target_crs)
#
#     # ✅ STEP 2: 创建 city_id 和 Shape_Area
#     city_gdf = city_gdf.reset_index(drop=True)
#     city_gdf['city_id'] = city_gdf.index
#     # city_gdf['Shape_Area'] = city_gdf.geometry.area
#     city_gdf['lst'] = 0.0  # 初始化
#
#     # ✅ STEP 3: 找出每个 grid 内完全包含的 clipped polygon
#     for idx, row in city_gdf.iterrows():
#         grid_geom = row.geometry
#         inside = clipped[clipped.within(grid_geom)]
#         if not inside.empty:
#             area_sum = inside.geometry.area.sum()
#             city_gdf.at[idx, 'lst'] = area_sum
#
#     # ✅ STEP 4: 计算 lst_ratio
#     city_gdf['lst_ratio'] = city_gdf['lst'] / city_gdf['Shape_Area']
#
#     # ✅ STEP 5: 导出为 ITRF_2000_UTM_K
#     itrf_proj = (
#         '+proj=tmerc +lat_0=38 +lon_0=127.5 '
#         '+k=0.9996 +x_0=1000000 +y_0=2000000 '
#         '+ellps=GRS80 +units=m +no_defs'
#     )
#     city_gdf_final = city_gdf.to_crs(itrf_proj)
#     city_gdf_final.to_file(output_path, driver='ESRI Shapefile')
#
#     print(f"✅ 完全包含统计完成并保存: {output_path}")
#
#
#
#
# # === 设置路径 ===
# # 所有 clipped shapefiles（一个 per grid）
# clipped_folder = r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles_clipped_with_fid'
# clipped_files = [f for f in os.listdir(clipped_folder) if f.endswith('.shp')]
#
# # 所有 grid shapefiles
# grid_folder = r'D:\seoul\grids'
# grid_files = [f for f in os.listdir(grid_folder) if f.endswith('.shp')]
#
# # 输出文件夹
# output_folder = r'D:\seoul\grids\city_ratio_outputs'
# os.makedirs(output_folder, exist_ok=True)
#
# def process_pair(grid_file):
#     try:
#         grid_name = os.path.splitext(grid_file)[0]
#         grid_path = os.path.join(grid_folder, grid_file)
#
#         matched_clipped = [
#             f for f in clipped_files if grid_name in f
#         ]
#
#         if not matched_clipped:
#             print(f"⚠️ 找不到与 {grid_file} 匹配的 clipped 文件")
#             return
#
#         clipped_path = os.path.join(clipped_folder, matched_clipped[0])
#         clipped_gdf = gpd.read_file(clipped_path)
#         clipped_gdf = clipped_gdf.reset_index(drop=True)
#         clipped_gdf['city_id'] = clipped_gdf['FID'] if 'FID' in clipped_gdf.columns else clipped_gdf.index
#
#         output_path = os.path.join(output_folder, f'city2020_lst_ratio_{grid_name}.shp')
#         if os.path.exists(output_path):
#             print(f"⏩ 已存在，跳过: {output_path}")
#             return
#
#         add_lst_ratio_within_only(clipped_gdf, grid_path, output_path)
#
#     except Exception as e:
#         print(f"❌ 处理 {grid_file} 失败: {e}")
#
# # 🚀 并行处理
# print("🚀 开始批量匹配 grid ↔ clipped...")
# with ThreadPoolExecutor(max_workers=4) as executor:
#     executor.map(process_pair, grid_files)

🚀 开始批量匹配 grid ↔ clipped...
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_1080m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_360m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_240m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_600m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_480m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_120m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_840m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_960m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_720m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_450m.shp
